# DCA Buy-the-Bear Monthly BTCUSDT Backtest

Buy $10 BTC every red monthly candle close. Sell 70% at every new all-time high monthly close.

**Strategy:**
- Entry: Monthly close < monthly open → buy $10 BTC at close
- Exit: Monthly close > all previous monthly closes (new ATH) → sell 70% of holdings at close
- Always buy on red candles regardless of position or recent sell

In [ ]:
import pandas as pd
import requests
import time
import json
from datetime import datetime, timezone

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: '%.8f' % x)

In [ ]:
# Fetch monthly BTCUSDT klines from Binance

def fetch_monthly_klines(symbol='BTCUSDT', limit=500):
    """Fetch all monthly klines from Binance spot."""
    url = 'https://api.binance.com/api/v3/klines'
    all_klines = []
    start_time = 0
    
    while True:
        params = {
            'symbol': symbol,
            'interval': '1M',
            'startTime': start_time,
            'limit': limit
        }
        resp = requests.get(url, params=params)
        data = resp.json()
        if not data or len(data) == 0:
            break
        all_klines.extend(data)
        print(f'Fetched {len(data)} months, up to {datetime.fromtimestamp(data[-1][0]/1000, tz=timezone.utc).strftime("%Y-%m")}')
        if len(data) < limit:
            break
        start_time = data[-1][0] + 1
        time.sleep(0.1)
    
    print(f'Total months fetched: {len(all_klines)}')
    return all_klines

klines = fetch_monthly_klines()

In [ ]:
# Convert to DataFrame

columns = [
    'timestamp', 'open', 'high', 'low', 'close', 'volume',
    'close_time', 'quote_vol', 'trades', 'taker_buy_base',
    'taker_buy_quote', 'ignore'
]

df = pd.DataFrame(klines, columns=columns)
df['date'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
for c in ['open', 'high', 'low', 'close', 'volume']:
    df[c] = df[c].astype(float)
df = df[['date', 'open', 'high', 'low', 'close', 'volume']].copy()
df = df.sort_values('date').reset_index(drop=True)
print(f'{df["date"].min().strftime("%Y-%m")} → {df["date"].max().strftime("%Y-%m")} ({len(df)} months)')
df.tail(3)

In [ ]:
# Strategy state variables

btc_held = 0.0
usdt_reserve = 0.0
total_invested = 0.0
highest_close = 0.0

records = []

for i, row in df.iterrows():
    close = row['close']
    month = row['date']
    action = 'nothing'
    
    # 1. Entry: buy $10 on red monthly candle
    if close < row['open']:
        btc_bought = 10.0 / close
        btc_held += btc_bought
        total_invested += 10.0
        usdt_reserve -= 10.0  # spent USDT
        action = f'buy $10 @ {close:.2f} (+{btc_bought:.8f} BTC)'
    
    # 2. Update highest close before checking sell
    prev_highest = highest_close
    if close > highest_close:
        highest_close = close
    
    # 3. Exit: sell 70% at new all-time high
    #    (only if we have BTC AND this candle actually set a new ATH)
    if btc_held > 0.000001 and close > prev_highest:
        sell_btc = btc_held * 0.7
        sell_usdt = sell_btc * close
        btc_held -= sell_btc
        usdt_reserve += sell_usdt
        act = f'sell 70% @ {close:.2f} (-{sell_btc:.8f} BTC, +{sell_usdt:.2f} USDT)'
        action = f'{action} | {act}' if 'buy' in action else act
    
    portfolio_value = btc_held * close + usdt_reserve
    
    records.append({
        'date': month,
        'close': close,
        'action': action,
        'btc_held': btc_held,
        'usdt_reserve': usdt_reserve,
        'total_invested': total_invested,
        'portfolio_value': portfolio_value,
    })

results = pd.DataFrame(records)
print(f'Total months: {len(results)}')
print(f'Trades with action: {len(results[results["action"] != "nothing"])}')
results.tail(5)

In [ ]:
# Summary

final = results.iloc[-1]
last_close = final['close']

print('='*60)
print('PORTFOLIO SUMMARY')
print('='*60)
print(f'Total invested:        {final["total_invested"]:.2f} USDT')
print(f'BTC held:              {final["btc_held"]:.8f} BTC')
print(f'BTC held value:        {final["btc_held"] * last_close:.2f} USDT (at {last_close:.2f})')
print(f'USDT reserve:          {final["usdt_reserve"]:.2f} USDT')
print(f'Portfolio value:       {final["portfolio_value"]:.2f} USDT')
print(f'Total P&L:             {final["portfolio_value"] - final["total_invested"]:.2f} USDT')
print(f'Return:                {((final["portfolio_value"] / final["total_invested"]) - 1) * 100:.2f}%')
print(f'Months:                {len(results)}')

# Count buys and sells
buys = [r for r in records if 'buy' in r['action']]
sells = [r for r in records if 'sell' in r['action']]
print(f'Buy months:            {len(buys)}')
print(f'Sell months:           {len(sells)}')

In [ ]:
# All trades with actions

pd.set_option('display.max_rows', None)
with_actions = results[results['action'] != 'nothing'].copy()
with_actions[['date', 'close', 'action', 'btc_held', 'usdt_reserve', 'total_invested', 'portfolio_value']]